In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
JupyterDash.infer_jupyter_proxy_config()
import dash_leaflet as dl
from dash import dcc, html, dash_table

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
import plotly.express as px
from dash.dependencies import Input, Output, State
import base64
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the CRUD Python module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Connect to database via CRUD Module
username = "aacuser"
password = "cs340_12345"
db = AnimalShelter(username, password)

# Read all documents to initialize the dataframe
df = pd.DataFrame.from_records(db.read({}))

# Drop the '_id' column to prevent Dash serialization crashes
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # Ensure this matches your uploaded logo filename exactly
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
    image_src = 'data:image/png;base64,{}'.format(encoded_image.decode())
except FileNotFoundError:
    # Fallback if the logo is missing so the app doesn't crash
    image_src = ''

app.layout = html.Div([
    # Logo and Unique Identifier Header
    html.Div([
        html.Img(src=image_src, style={'width': '250px', 'display': 'inline-block'}),
        html.Center(html.B(html.H1('Grazioso Salvare Dashboard - Daniel', style={'display': 'inline-block', 'marginLeft': '30px'})))
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'}),
    
    html.Hr(),
    
    # Interactive filtering options (Radio Buttons)
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'Water'},
                {'label': ' Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': ' Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': ' Reset - Display All', 'value': 'Reset'}
            ],
            value='Reset', # Default selection
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        ),
        style={'display': 'flex', 'justifyContent': 'center', 'padding': '10px'}
    ),
    
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        row_selectable="single", # Required for the map callback
        selected_rows=[0],       # Defaults to selecting the first row
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'}
    ),
    
    html.Br(),
    html.Hr(),
    
    # Side-by-side Graph and Map layout
    html.Div(className='row', style={'display' : 'flex'}, children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'width': '50%'}
        ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'width': '50%'}
        )
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback 1: Filter the interactive data table
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Determine the query based on the selected radio button
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Bloodhound"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    else:
        query = {} # Reset returns all records

    # Fetch new data and format it
    filtered_data = db.read(query)
    filtered_df = pd.DataFrame.from_records(filtered_data)
    
    if not filtered_df.empty and '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
    elif filtered_df.empty:
        return []
        
    return filtered_df.to_dict('records')

# Callback 2: Display the breeds of animal via Pie Chart based on filtered table data
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None:
        return []
        
    dff = pd.DataFrame.from_dict(viewData)
    
    if dff.empty:
        return []
        
    # Create a pie chart showing the distribution of breeds currently in the table
    return [
        dcc.Graph(            
            figure = px.pie(dff, names='breed', title='Preferred Animals by Breed')
        )    
    ]
    
# Callback 3: Highlight a cell on the data table when selected
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Callback 4: Update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  
    if viewData is None:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return []
        
    # Default to the first row if none selected
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    # Safely extract coordinate and animal data
    try:
        lat = float(dff.iloc[row]['location_lat'])
        lon = float(dff.iloc[row]['location_long'])
        animal_breed = str(dff.iloc[row]['breed'])
        animal_name = str(dff.iloc[row]['name'])
    except:
        # Fallback to Austin TX center if data is missing or out of bounds
        lat = 30.2672
        lon = -97.7431
        animal_breed = "Unknown"
        animal_name = "Unknown"

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(animal_breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(animal_name)
                ])
            ])
        ])
    ]

# Run app and display inline
app.run_server(mode='inline', port=8055)